In [0]:
data = [
    ("Kochi", "2024-01", 120000),
    ("Kochi", "2024-02", 135000),
    ("Kochi", "2024-03", 128000),
    ("Kochi", "2024-04", 150000),
    ("Chennai", "2024-01", 110000),
    ("Chennai", "2024-02", 125000),
    ("Chennai", "2024-03", 140000),
    ("Chennai", "2024-04", 138000),
    ("Bangalore", "2024-01", 150000),
    ("Bangalore", "2024-02", 155000),
    ("Bangalore", "2024-03", 149000),
    ("Bangalore", "2024-04", 165000),
    ("Hyderabad", "2024-01", 90000),
    ("Hyderabad", "2024-02", 105000),
    ("Hyderabad", "2024-03", 112000),
    ("Hyderabad", "2024-04", 125000),
    ("Delhi", "2024-01", 180000),
    ("Delhi", "2024-02", 175000),
    ("Delhi", "2024-03", 190000),
    ("Delhi", "2024-04", 210000)
]
 
columns = [
    "city",
    "month",
    "sales_amount"
]
 
df_sales = spark.createDataFrame(data, columns)
df_sales.show()
 

+---------+-------+------------+
|     city|  month|sales_amount|
+---------+-------+------------+
|    Kochi|2024-01|      120000|
|    Kochi|2024-02|      135000|
|    Kochi|2024-03|      128000|
|    Kochi|2024-04|      150000|
|  Chennai|2024-01|      110000|
|  Chennai|2024-02|      125000|
|  Chennai|2024-03|      140000|
|  Chennai|2024-04|      138000|
|Bangalore|2024-01|      150000|
|Bangalore|2024-02|      155000|
|Bangalore|2024-03|      149000|
|Bangalore|2024-04|      165000|
|Hyderabad|2024-01|       90000|
|Hyderabad|2024-02|      105000|
|Hyderabad|2024-03|      112000|
|Hyderabad|2024-04|      125000|
|    Delhi|2024-01|      180000|
|    Delhi|2024-02|      175000|
|    Delhi|2024-03|      190000|
|    Delhi|2024-04|      210000|
+---------+-------+------------+



In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
 


#  Level 1 — Running Totals
## Task 1
Calculate running sales amount for each city.

Expected Output:
City |	Month |	Sales |	Running Total

In [0]:
df_sales = df_sales.withColumnRenamed("sales_amount", "sales")
df_sales.withColumn("Running Total", sum("sales").over(Window.partitionBy("city").orderBy("month"))).show()

+---------+-------+------+-------------+
|     city|  month| sales|Running Total|
+---------+-------+------+-------------+
|Bangalore|2024-01|150000|       150000|
|Bangalore|2024-02|155000|       305000|
|Bangalore|2024-03|149000|       454000|
|Bangalore|2024-04|165000|       619000|
|  Chennai|2024-01|110000|       110000|
|  Chennai|2024-02|125000|       235000|
|  Chennai|2024-03|140000|       375000|
|  Chennai|2024-04|138000|       513000|
|    Delhi|2024-01|180000|       180000|
|    Delhi|2024-02|175000|       355000|
|    Delhi|2024-03|190000|       545000|
|    Delhi|2024-04|210000|       755000|
|Hyderabad|2024-01| 90000|        90000|
|Hyderabad|2024-02|105000|       195000|
|Hyderabad|2024-03|112000|       307000|
|Hyderabad|2024-04|125000|       432000|
|    Kochi|2024-01|120000|       120000|
|    Kochi|2024-02|135000|       255000|
|    Kochi|2024-03|128000|       383000|
|    Kochi|2024-04|150000|       533000|
+---------+-------+------+-------------+



## Task 2
Find total cumulative sales till each month.


In [0]:
df_sales.groupBy("month").agg(sum("sales").alias("monthly_sales")).withColumn("cummulative sales", sum("monthly_sales").over(Window.orderBy("month"))).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-------+-------------+-----------------+
|  month|monthly_sales|cummulative sales|
+-------+-------------+-----------------+
|2024-01|       650000|           650000|
|2024-02|       695000|          1345000|
|2024-03|       719000|          2064000|
|2024-04|       788000|          2852000|
+-------+-------------+-----------------+



#  Level 2 — Lag Analysis
## Task 3
Create column:
previous_month_sales
using lag().

In [0]:
df_pre = df_sales.withColumn("previous_month_sales", lag("sales").over(Window.partitionBy("city").orderBy("month")))
df_pre.show()

+---------+-------+------+--------------------+
|     city|  month| sales|previous_month_sales|
+---------+-------+------+--------------------+
|Bangalore|2024-01|150000|                NULL|
|Bangalore|2024-02|155000|              150000|
|Bangalore|2024-03|149000|              155000|
|Bangalore|2024-04|165000|              149000|
|  Chennai|2024-01|110000|                NULL|
|  Chennai|2024-02|125000|              110000|
|  Chennai|2024-03|140000|              125000|
|  Chennai|2024-04|138000|              140000|
|    Delhi|2024-01|180000|                NULL|
|    Delhi|2024-02|175000|              180000|
|    Delhi|2024-03|190000|              175000|
|    Delhi|2024-04|210000|              190000|
|Hyderabad|2024-01| 90000|                NULL|
|Hyderabad|2024-02|105000|               90000|
|Hyderabad|2024-03|112000|              105000|
|Hyderabad|2024-04|125000|              112000|
|    Kochi|2024-01|120000|                NULL|
|    Kochi|2024-02|135000|              

## Task 4
Calculate:
sales_difference
Current Month − Previous Month

In [0]:
df_diff = df_pre.withColumn("sales_difference",col("sales")-col("previous_month_sales"))
df_diff.show()

+---------+-------+------+--------------------+----------------+
|     city|  month| sales|previous_month_sales|sales_difference|
+---------+-------+------+--------------------+----------------+
|Bangalore|2024-01|150000|                NULL|            NULL|
|Bangalore|2024-02|155000|              150000|            5000|
|Bangalore|2024-03|149000|              155000|           -6000|
|Bangalore|2024-04|165000|              149000|           16000|
|  Chennai|2024-01|110000|                NULL|            NULL|
|  Chennai|2024-02|125000|              110000|           15000|
|  Chennai|2024-03|140000|              125000|           15000|
|  Chennai|2024-04|138000|              140000|           -2000|
|    Delhi|2024-01|180000|                NULL|            NULL|
|    Delhi|2024-02|175000|              180000|           -5000|
|    Delhi|2024-03|190000|              175000|           15000|
|    Delhi|2024-04|210000|              190000|           20000|
|Hyderabad|2024-01| 90000

## Task 5
Identify months where sales decreased.


In [0]:
df_diff.filter(col("sales_difference")<0).show()

+---------+-------+------+--------------------+----------------+
|     city|  month| sales|previous_month_sales|sales_difference|
+---------+-------+------+--------------------+----------------+
|Bangalore|2024-03|149000|              155000|           -6000|
|  Chennai|2024-04|138000|              140000|           -2000|
|    Delhi|2024-02|175000|              180000|           -5000|
|    Kochi|2024-03|128000|              135000|           -7000|
+---------+-------+------+--------------------+----------------+



#  Level 3 — Growth Analysis
## Task 6
Calculate:
growth_percentage
Formula:
(Current Sales - Previous Sales)
/ Previous Sales * 100

In [0]:
df_grow = df_diff.withColumn("growth_percetage",round((col("sales_difference"))/col("previous_month_sales")*100, 2))
df_grow.show()

+---------+-------+------+--------------------+----------------+----------------+
|     city|  month| sales|previous_month_sales|sales_difference|growth_percetage|
+---------+-------+------+--------------------+----------------+----------------+
|Bangalore|2024-01|150000|                NULL|            NULL|            NULL|
|Bangalore|2024-02|155000|              150000|            5000|            3.33|
|Bangalore|2024-03|149000|              155000|           -6000|           -3.87|
|Bangalore|2024-04|165000|              149000|           16000|           10.74|
|  Chennai|2024-01|110000|                NULL|            NULL|            NULL|
|  Chennai|2024-02|125000|              110000|           15000|           13.64|
|  Chennai|2024-03|140000|              125000|           15000|            12.0|
|  Chennai|2024-04|138000|              140000|           -2000|           -1.43|
|    Delhi|2024-01|180000|                NULL|            NULL|            NULL|
|    Delhi|2024-

## Task 7
Find highest growth month for each city.

In [0]:
df_grow.withColumn("rank", dense_rank().over(Window.partitionBy("city").orderBy(col("growth_percetage").desc()))).filter(col("rank")<2).show()

+---------+-------+------+--------------------+----------------+----------------+----+
|     city|  month| sales|previous_month_sales|sales_difference|growth_percetage|rank|
+---------+-------+------+--------------------+----------------+----------------+----+
|Bangalore|2024-04|165000|              149000|           16000|           10.74|   1|
|  Chennai|2024-02|125000|              110000|           15000|           13.64|   1|
|    Delhi|2024-04|210000|              190000|           20000|           10.53|   1|
|Hyderabad|2024-02|105000|               90000|           15000|           16.67|   1|
|    Kochi|2024-04|150000|              128000|           22000|           17.19|   1|
+---------+-------+------+--------------------+----------------+----------------+----+




## Task 8
Find city with highest growth percentage overall.


In [0]:
df_grow_per = df_grow.orderBy(desc(col("growth_percetage")))
df_grow_per.limit(5).show()
print("city with highest growth percentage overall is ", df_grow_per.collect()[0][0])

+---------+-------+------+--------------------+----------------+----------------+
|     city|  month| sales|previous_month_sales|sales_difference|growth_percetage|
+---------+-------+------+--------------------+----------------+----------------+
|    Kochi|2024-04|150000|              128000|           22000|           17.19|
|Hyderabad|2024-02|105000|               90000|           15000|           16.67|
|  Chennai|2024-02|125000|              110000|           15000|           13.64|
|    Kochi|2024-02|135000|              120000|           15000|            12.5|
|  Chennai|2024-03|140000|              125000|           15000|            12.0|
+---------+-------+------+--------------------+----------------+----------------+

city with highest growth percentage overall is  Kochi


#  Level 4 — Ranking
## Task 9
Rank all city-month combinations by sales amount.

In [0]:
df_sales.withColumn("rank", dense_rank().over(Window.orderBy(desc(col("sales"))))).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-------+------+----+
|     city|  month| sales|rank|
+---------+-------+------+----+
|    Delhi|2024-04|210000|   1|
|    Delhi|2024-03|190000|   2|
|    Delhi|2024-01|180000|   3|
|    Delhi|2024-02|175000|   4|
|Bangalore|2024-04|165000|   5|
|Bangalore|2024-02|155000|   6|
|    Kochi|2024-04|150000|   7|
|Bangalore|2024-01|150000|   7|
|Bangalore|2024-03|149000|   8|
|  Chennai|2024-03|140000|   9|
|  Chennai|2024-04|138000|  10|
|    Kochi|2024-02|135000|  11|
|    Kochi|2024-03|128000|  12|
|  Chennai|2024-02|125000|  13|
|Hyderabad|2024-04|125000|  13|
|    Kochi|2024-01|120000|  14|
|Hyderabad|2024-03|112000|  15|
|  Chennai|2024-01|110000|  16|
|Hyderabad|2024-02|105000|  17|
|Hyderabad|2024-01| 90000|  18|
+---------+-------+------+----+



## Task 10
Find top sales month for each city.

In [0]:
df_sales.withColumn("top_sales_month", dense_rank().over(Window.partitionBy("city").orderBy(desc(col("sales"))))).filter(col("top_sales_month")==1).show()


+---------+-------+------+---------------+
|     city|  month| sales|top_sales_month|
+---------+-------+------+---------------+
|Bangalore|2024-04|165000|              1|
|  Chennai|2024-03|140000|              1|
|    Delhi|2024-04|210000|              1|
|Hyderabad|2024-04|125000|              1|
|    Kochi|2024-04|150000|              1|
+---------+-------+------+---------------+



## Task 11
Find second highest sales month for each city.

In [0]:
df_sales.withColumn("top_sales_month", dense_rank().over(Window.partitionBy("city").orderBy(desc(col("sales"))))).filter(col("top_sales_month")==2).show()


+---------+-------+------+---------------+
|     city|  month| sales|top_sales_month|
+---------+-------+------+---------------+
|Bangalore|2024-02|155000|              2|
|  Chennai|2024-04|138000|              2|
|    Delhi|2024-03|190000|              2|
|Hyderabad|2024-03|112000|              2|
|    Kochi|2024-02|135000|              2|
+---------+-------+------+---------------+



# Level 5 — Business Analytics
## Task 12
Find city with highest total sales.

In [0]:
df_hi_sale = df_sales.groupBy("city").agg(sum("sales").alias("total_sales")).orderBy(desc("total_sales"))
df_hi_sale.show()
print("city with highest total sales is ", df_hi_sale.collect()[0][0])

+---------+-----------+
|     city|total_sales|
+---------+-----------+
|    Delhi|     755000|
|Bangalore|     619000|
|    Kochi|     533000|
|  Chennai|     513000|
|Hyderabad|     432000|
+---------+-----------+

city with highest total sales is  Delhi


## Task 13
Find city with most consistent growth.
(Hint: smallest fluctuations)

In [0]:
df_consistent = df_grow.groupBy("city").agg(round(stddev("growth_percetage")).alias("stddev_sales")).orderBy("stddev_sales")
df_consistent.show()
print(" city with most consistent growth is ", df_consistent.collect()[0][0])

+---------+------------+
|     city|stddev_sales|
+---------+------------+
|Hyderabad|         5.0|
|Bangalore|         7.0|
|    Delhi|         7.0|
|  Chennai|         8.0|
|    Kochi|        12.0|
+---------+------------+

 city with most consistent growth is  Hyderabad


## Task 14
Find city that experienced the largest sales drop.


In [0]:

df_diff.dropna().orderBy(col("sales_difference")).show(1)

+-----+-------+------+--------------------+----------------+
| city|  month| sales|previous_month_sales|sales_difference|
+-----+-------+------+--------------------+----------------+
|Kochi|2024-03|128000|              135000|           -7000|
+-----+-------+------+--------------------+----------------+
only showing top 1 row


# Challenge Tasks
## Task 15
Create final report:
| City | Month | Sales | Previous Sales | Difference | Growth % | Running Total |

In [0]:
final = df_grow.withColumn("running_total",sum("sales").over(Window.partitionBy("city").orderBy(col("month"))))
final.show()

+---------+-------+------+--------------------+----------------+----------------+-------------+
|     city|  month| sales|previous_month_sales|sales_difference|growth_percetage|running_total|
+---------+-------+------+--------------------+----------------+----------------+-------------+
|Bangalore|2024-01|150000|                NULL|            NULL|            NULL|       150000|
|Bangalore|2024-02|155000|              150000|            5000|            3.33|       305000|
|Bangalore|2024-03|149000|              155000|           -6000|           -3.87|       454000|
|Bangalore|2024-04|165000|              149000|           16000|           10.74|       619000|
|  Chennai|2024-01|110000|                NULL|            NULL|            NULL|       110000|
|  Chennai|2024-02|125000|              110000|           15000|           13.64|       235000|
|  Chennai|2024-03|140000|              125000|           15000|            12.0|       375000|
|  Chennai|2024-04|138000|              

## Task 16
Management wants to know:

"Which city should receive additional inventory investment?"

Provide:
analysis
reasoning
recommendation
 

In [0]:
final.groupBy("city").agg(sum("sales").alias("total_sales"),avg("growth_percetage").alias("avg_growth")).orderBy(desc("total_sales")).show()

+---------+-----------+-----------------+
|     city|total_sales|       avg_growth|
+---------+-----------+-----------------+
|    Delhi|     755000|             5.44|
|Bangalore|     619000|              3.4|
|    Kochi|     533000|8.166666666666666|
|  Chennai|     513000|             8.07|
|Hyderabad|     432000|            11.65|
+---------+-----------+-----------------+



### If the objective is maximize current revenue
Then Delhi is the best choice because:
- Highest total sales
- Highest monthly sales
- Largest market demand
- Lowest risk of unsold inventory



### If the objective is invest in the fastest-growing market
Then Hyderabad becomes the best choice because:
- Highest average growth rate (~11.65%)
- Positive growth every month
- Consistent upward trend
- Potential for future expansion
